# Claude Complaint Labeling: Sephora Moisturizer Reviews

This notebook uses Claude to label complaints in a stratified sample of moisturizer reviews. It covers:
1. Sample design: 8,000 reviews from the 20 most-reviewed moisturizers, 100 per product per skin type
2. Complaint categories: drafted from the Phase 2 keyword analysis and refined with Claude's suggestions
3. Pilot: labeling a small set of reviews and hand-checking the results before the full run
4. Full labeling: all 8,000 reviews through the Batch API
5. Quality check and API usage: agreement with hand labels, plus tokens and cost

Labels are saved to `Data/`, so the final report notebook can use them without calling the API again.

## Setup & Data Load

In [22]:

%load_ext autoreload
%autoreload 2

# Imports
import sys
sys.path.append("..")
import os
import json
import re
import numpy as np
import pandas as pd
import anthropic
from dotenv import load_dotenv
from src.utils import load_data
from sklearn.metrics import cohen_kappa_score


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
# Data Load
df = load_data()
df["full_text"] = (df["review_title"].fillna("") + ". " + df["review_text"]).str.lstrip(". ")
print(f"Loaded {len(df):,} reviews")

Loaded 184,664 reviews


In [39]:
# Connect to Claude and test the API key
load_dotenv("../.env", override=True)
client = anthropic.Anthropic()
MODEL = "claude-haiku-5-5"   # lowest-cost Claude model

usage_log = []   # (input_tokens, output_tokens) for every real-time call in this notebook


def ask_claude(prompt, system=None, max_tokens=1500):
    """Send one prompt to Claude with thinking off, log token usage,
    and return the text of the reply. (Temperature can't be set on current models.)"""
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "thinking": {"type": "disabled"},
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    usage_log.append((response.usage.input_tokens, response.usage.output_tokens))
    # Keep only the text parts of the reply (skips any thinking blocks)
    return "".join(block.text for block in response.content if block.type == "text")


print(ask_claude("Reply with only the word: connected", max_tokens=10))

connected


## 1. Sample Design

The analysis compares complaint rates across skin types within the same product, since shoppers already choose products partly by skin type (notebook 2). Products are eligible if they have at least 30 reviews in every skin type. From the 20 most-reviewed eligible moisturizers, 100 reviews are sampled per product per skin type, giving 8,000 reviews with equal group sizes. Two products are excluded and replaced by the next most-reviewed products because they aren't comparable to moisturizers: Bobbi Brown's Vitamin Enriched Face Base, which is sold partly as a makeup primer, and Farmacy's Honeymoon Glow AHA Resurfacing Night Serum, an exfoliating acid treatment whose complaints (e.g., stinging from the acids) reflect its product type rather than skin-type fit.

In [25]:
N_PER_GROUP = 100
MIN_PER_TYPE = 30
EXCLUDE = ["Vitamin Enriched Face Base Priming Moisturizer",
           "Honeymoon Glow AHA Resurfacing Night Serum"]
# Reviews per product and skin type
counts = (df.groupby(["product_id", "product_name", "brand_name", "skin_type"])
            .size().unstack(fill_value=0))
eligible = counts[(counts >= MIN_PER_TYPE).all(axis=1)]
eligible = eligible[~eligible.index.get_level_values("product_name").isin(EXCLUDE)]

top20 = eligible.assign(total=eligible.sum(axis=1)).sort_values("total", ascending=False).head(20)
top_ids = top20.index.get_level_values("product_id")

sample = (df[df["product_id"].isin(top_ids)]
            .groupby(["product_id", "skin_type"])
            .sample(N_PER_GROUP, random_state=42)
            .reset_index(drop=True))

print(f"Sample size: {len(sample):,}")
print(sample.groupby("skin_type").size())
display(top20.reset_index(level="product_id", drop=True))

sample.to_csv("../Data/labeling_sample.csv", index=False)

Sample size: 8,000
skin_type
combination    2000
dry            2000
normal         2000
oily           2000
dtype: int64


,skin_type,combination,dry,normal,oily,total
product_name,brand_name,,,,,
The True Cream Aqua Bomb,belif,3525,1074,796,885,6280
Protini Polypeptide Firming Refillable Moisturizer,Drunk Elephant,3367,1159,774,659,5959
Ultra Repair Cream Intense Hydration,First Aid Beauty,2702,1822,578,375,5477
The True Cream Moisturizing Bomb,belif,2107,1072,451,324,3954
The Water Cream Oil-Free Pore Minimizing Moisturizer,Tatcha,2238,469,421,760,3888
Magic Cream Moisturizer with Hyaluronic Acid,Charlotte Tilbury,1804,722,549,370,3445
Santorini Grape Poreless Skin Cream,KORRES,1857,511,460,322,3150
The Dewy Skin Cream Plumping & Hydrating Moisturizer,Tatcha,1434,1068,414,179,3095
Greek Yoghurt Nourishing Probiotic Gel-Cream,KORRES,1751,517,393,412,3073


## 2. Complaint Categories

A draft list of categories comes from the Phase 2 keyword analysis. To check that it isn't missing major complaint types, Claude is first asked to suggest categories from 40 sampled reviews rated 1–3 stars (10 per skin type).

In [26]:
# Sample Prompt 1: CLaude Suggests Categories
discovery = (sample[sample["rating"] <= 3]
             .groupby("skin_type")
             .sample(10, random_state=7))
numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(discovery["full_text"]))

discovery_prompt = f"""Below are {len(discovery)} customer reviews of facial moisturizers from Sephora.

Identify the main types of complaints reviewers make about the products. Suggest 8 to 10
complaint categories that together cover most complaints. For each category, give:
- a short name in lowercase with underscores instead of spaces (e.g., greasy_heavy)
- a one-sentence definition
- one example phrase quoted from the reviews

Only include problems with the product itself, not reviewers describing their own skin.

Reviews:
{numbered}"""

print(ask_claude(discovery_prompt))

Here are 9 complaint categories that together cover most of the product-focused complaints in these reviews.

**1. breakouts_clogged_pores**
Reviewer reports that the product caused new breakouts, bumps, or clogged pores.
Example: "it made me break out little tiny bumps all over my face"

**2. greasy_heavy_texture**
The product feels thick, greasy, or heavy on the skin, or sits on top rather than absorbing.
Example: "it just feels greasy and sticky"

**3. poor_absorption**
The product does not sink into the skin and leaves a surface film or residue that persists over time.
Example: "it sits on the surface for a long time, rather than being absorbed"

**4. insufficient_hydration**
The product fails to provide lasting moisture, leaving skin feeling dry, tight, or unmoisturized shortly after application.
Example: "my skin feels so dry! Like I feel like it made my skin feel worse than before"

**5. strong_fragrance_odor**
The scent is overpowering, sweet, petroleum-like, or otherwise unple

**Observations**

Claude's suggested categories largely match the draft built from the Phase 2 keyword analysis (breakouts, greasiness, dryness, fragrance, irritation, pilling, price, and lack of results), which supports both lists. The final categories (Cell 9) reflect three refinements:
- Separated texture from pilling: Claude suggested a separate "poor absorption" category for products that leave a film or residue. This overlapped with both greasiness and pilling in the draft definitions, so the boundaries were redrawn: **greasy_heavy** covers how a product feels on the skin (including not absorbing or leaving a film), while **pilling** covers how it interacts with makeup and other products.
- Kept "no results": Claude's "underwhelming results" category matches the draft's no_results category and the "not work" signal from the keyword analysis.
- Kept packaging and reformulation: These did not appear in this sample of 40 reviews but were suggested in an earlier run on a preliminary sample. They are kept as categories and can be merged into "other" if they turn out to be rare in the full labeling.

Claude's examples also contained an error: "the metals (aka tin oxide) in this product made me break out" was given as an example of irritation, but it describes a breakout. This is a reminder that Claude's output needs to be checked, which the pilot and spot-check below are designed to do.

In [27]:
# Final categories: Phase 2 keyword draft, refined with Claude's suggestions (Cell 7)
CATEGORIES = {
    "greasy_heavy": "Feels greasy, heavy, thick, or sticky; doesn't absorb; or leaves a film, shine, or oily skin",
    "breakouts": "Caused acne, breakouts, whiteheads, bumps, or clogged pores",
    "not_moisturizing": "Not moisturizing or hydrating enough; skin still felt dry, flaky, or tight",
    "irritation": "Caused redness, stinging, burning, itching, or another skin reaction",
    "fragrance": "Unpleasant, odd, or too-strong scent",
    "pilling": "Pills or balls up, or makes makeup, sunscreen, or other products separate or slide",
    "no_results": "Didn't deliver visible results or the promised benefits",
    "price_value": "Too expensive for its performance or amount, or cheaper alternatives work as well",
    "packaging": "Problems with the jar, pump, lid, or container",
    "reformulation": "A newer formula performs worse than the version the reviewer liked before",
    "other": "A clear complaint that fits none of the categories above",
}
print(f"{len(CATEGORIES)} categories")

11 categories


## 3. Labeling Prompt and Pilot

Each review is sent to Claude with instructions listing the complaint categories and rules for applying them. The rules target the cases where keyword counts and VADER failed in Phase 2: a review can contain several complaints or none, mild complaints in positive reviews still count, and problems the product avoided ("doesn't break me out"), problems with other products, and reviewers describing their own skin type are not complaints. Claude replies in JSON so labels can be read directly into a table.

Before labeling the full sample, the prompt is tested on 30 reviews (6 per star rating). These reviews are labeled by hand before seeing Claude's labels, so the comparison isn't influenced by Claude's answers. Agreement is measured with Cohen's kappa, which corrects for agreement expected by chance. The full sample is labeled only if kappa is at least 0.6, the conventional threshold for substantial agreement; otherwise the prompt is revised and tested on a fresh set of reviews.

In [28]:
# Format the categories as a bulleted list for the prompt
category_list = "\n".join(f"- {name}: {desc}" for name, desc in CATEGORIES.items())

# Instructions sent with every review: categories, labeling rules, and the required JSON format
LABEL_SYSTEM_PROMPT = f"""You label customer reviews of facial moisturizers sold at Sephora.

List every complaint the reviewer makes about THIS product, using only these categories:
{category_list}

Rules:
- A review can have several complaints, or none.
- Mild complaints in positive reviews still count (e.g., "love it, but the smell is strong").
- Do NOT count:
  - problems the product avoided or fixed ("doesn't break me out", "finally not greasy")
  - problems with other products the reviewer compares it to
  - the reviewer describing their own skin type or skin concerns ("I have oily, acne-prone skin")
- Use "other" only for a clear complaint that fits none of the other categories.

Respond with JSON only, in exactly this format: {{"complaints": ["category_name", ...]}}
Use an empty list if there are no complaints."""


def parse_labels(reply):
    """Read Claude's JSON reply into a list of valid category names, or None if it can't be read."""
    # Use the last {...} in the reply, so any explanation Claude writes before the JSON is ignored
    matches = re.findall(r"\{[^{}]*\}", reply)
    if not matches:
        return None   # no complete JSON object (e.g., the reply was cut off)
    try:
        labels = json.loads(matches[-1])["complaints"]
        # Keep only names from our category list, dropping any Claude invented
        return [c for c in labels if c in CATEGORIES]
    except (KeyError, json.JSONDecodeError):
        return None

# Token limit for each label reply: high enough that a short explanation before the JSON
# doesn't cut off the answer (100 tokens truncated 1 of 30 test replies)
LABEL_MAX_TOKENS = 300

def label_review(text):
    """Label one review in real time (used for the test set; the full sample uses the Batch API)."""
    return parse_labels(ask_claude(text, system=LABEL_SYSTEM_PROMPT, max_tokens=LABEL_MAX_TOKENS))

In [37]:
# Pilot sample: 6 reviews per star rating, so positive, mixed, and negative reviews are all tested
pilot = sample.groupby("rating").sample(6, random_state=21).copy()

# Label each pilot review with Claude
pilot["claude_labels"] = pilot["full_text"].apply(label_review)

# Show labels as readable text: "no complaint" for an empty list, "UNREADABLE" if parsing failed
pd.set_option("display.max_colwidth", 500)
pilot_view = pilot.assign(
    labels=pilot["claude_labels"].apply(
        lambda x: "UNREADABLE" if x is None else (", ".join(x) if x else "no complaint"))
).reset_index(drop=True)
display(pilot_view[["rating", "skin_type", "full_text", "labels"]])

# Check for parsing failures
print("Unreadable replies:", pilot["claude_labels"].isna().sum())

# Average tokens per review, used to estimate the full batch cost
pilot_calls = usage_log[-len(pilot):]
print(f"Average tokens per review: {np.mean([u[0] for u in pilot_calls]):.0f} input, "
      f"{np.mean([u[1] for u in pilot_calls]):.0f} output")

,rating,skin_type,full_text,labels
0,1,oily,"Say yes if you love having a greasy film. I took Sephora’s skin quiz. This was one of the products recommended to me. I looked at the rating...yup, pretty good ones. Yet, this product did not work for my oily skin. I used the whole bottle, just to see if it would grow on me. Nope. Honestly, I don’t understand why the word gel is even used in the name. It’s more cream than gel. When I rub it on my face, it leaves a greasy film that stays on all day. If I powder my face, that greasy film will ...","greasy_heavy, fragrance, other"
1,1,normal,Too Heavy. Did not like this.....Too heavy and greasy.,greasy_heavy
2,1,normal,"Dries out and tightens skin. I received a trial of this complimentary through Influenster. I have normal skin but it did not react well to this. My skin is sensitive and it caused my face to dry out and tighten. I know it was this product, because as soon as I felt dryness I discontinued use of it and only using my normal cleanser and moisturizer. I would not recommend this product as it left a stinging/tingling feeling when applied and dried my skin out very quickly.","not_moisturizing, irritation"
3,1,dry,Broke me out in a day! It’s very sticky and I broke out all over my face within a day. From the reviews I can see that it happens to a lot of other people. Stay away if you have sensitive skin.,"breakouts, greasy_heavy, irritation"
4,1,dry,Not for dry skin!. I purchased this for my dry skin. It literally felt like I had put nothing on my face even when I applied quite a fair amount. No change to my skin at all. Probably isn’t the best for dry skin. My skin isn’t even super dry!,"not_moisturizing, no_results"
5,1,normal,"not for me, waxy and sits on skin. I was excited to try this product in a sample pack. I used it in the morning for three days, in hopes that my first day’s observations would subside. They did not. This is a waxy, sit on your skin formula, that never seemed to be absorbed by my skin. At the end of the day, I felt like I had to wash my face due to wanting to wash this product off. I had no other changes to my normal routine during this time.",greasy_heavy
6,2,normal,"I really wanted to love this. I really wanted to love this, as I’ve been trying to use clean skincare, and have quite a few DE products. Sadly, every time I add Protini, I end up breaking out. My skin isn’t overly sensitive and can handle actives, but for some reason this just simply doesn’t work for me. I’ve tried it alone, every other day, cocktail with LaLa, and regardless of all of these, I wake up with breakouts. Will probably have to use as a neck cream. Total bummer.",breakouts
7,2,oily,Worst break-outs ever. When I first started using this product I loved the formula. My skin was soft and moisturized. Over time though I had really bad sustained acne. I figured it was something else so I didn’t change my moisturizer. I started using a different moisturizer and my skin cleared up. I tried this one again later then the same deep acne came back.,breakouts
8,2,normal,This doesn’t sink into my skin quick enough and it doesn’t hydrate enough. I regret buying it,"greasy_heavy, not_moisturizing"
9,2,dry,"I received this product from influenster to try. I don’t necessarily dislike this product, but I don’t love it and I don’t really think I’d ever buy it. Using it in the morning in place of my regular moisturizer it felt too sticky after application and then tightened. When I switched it to my night routine it didn’t feel moisturizing enough.","greasy_heavy, not_moisturizing"


Unreadable replies: 0
Average tokens per review: 708 input, 22 output


In [38]:
# Test set: 6 reviews per star rating, so positive, mixed, and negative reviews are all included
pilot = sample.groupby("rating").sample(6, random_state=21).reset_index(drop=True)

# Export review text only (no ratings or Claude labels), with a blank column for hand labels.
# Skip the export if the file already exists, so rerunning the notebook never erases hand labels.
HAND_LABEL_PATH = "../Data/pilot_hand_labels.csv"
if os.path.exists(HAND_LABEL_PATH):
    print(f"{HAND_LABEL_PATH} already exists, not overwritten")
else:
    pilot[["review_id", "full_text"]].assign(my_labels="").to_csv(HAND_LABEL_PATH, index=False)
    print(f"Exported {len(pilot)} reviews for hand labeling")

../Data/pilot_hand_labels.csv already exists, not overwritten


In [31]:
# Load hand labels, then attach the original review text from the sample
# (Excel can garble curly apostrophes, so the text column from the spreadsheet isn't used;
#  utf-8-sig handles the hidden marker Excel adds to the start of CSV UTF-8 files)
hand = (pd.read_csv(HAND_LABEL_PATH, dtype={"review_id": str}, encoding="utf-8-sig")
          [["review_id", "my_labels"]]
          .merge(pilot[["review_id", "full_text"]], on="review_id", how="left"))
print("Reviews matched to the original text:", hand["full_text"].notna().sum(), "of", len(hand))

# Convert "a, b" text into lists; "none", "no_complaint", or blank means no complaint
hand["hand_labels"] = hand["my_labels"].fillna("none").apply(
    lambda s: [] if s.strip().lower() in {"none", "no_complaint", ""}
    else [c.strip().lower() for c in s.split(",")])

# Catch typos in hand labels before comparing
typos = {c for labels in hand["hand_labels"] for c in labels if c not in CATEGORIES}
print("Unrecognized hand labels (fix in the CSV if any):", typos or "none")

# Claude labels the same 30 reviews
hand["claude_labels"] = hand["full_text"].apply(label_review)
print("Unreadable Claude replies:", hand["claude_labels"].isna().sum())

# Convert each review's labels into yes/no for every category (rows = reviews, columns = categories)
cats = list(CATEGORIES)
def to_matrix(label_lists):
    return np.array([[int(c in (labels or [])) for c in cats] for labels in label_lists])
H, C = to_matrix(hand["hand_labels"]), to_matrix(hand["claude_labels"])

# Overall agreement: kappa across every review x category decision, plus exact match per review
kappa = cohen_kappa_score(H.ravel(), C.ravel())
exact = (H == C).all(axis=1).mean()
print(f"\nCohen's kappa (all {H.size} review x category decisions): {kappa:.2f}")
print(f"Exact match (identical labels for the whole review): {exact:.0%}")
print("PASS: kappa >= 0.6, ready for the full batch" if kappa >= 0.6
      else "BELOW 0.6: revise the prompt and test on a new set")

# Per-category breakdown: how often each rater used the category and how often they agreed
per_cat = pd.DataFrame({
    "hand_yes": H.sum(axis=0),
    "claude_yes": C.sum(axis=0),
    "both_yes": (H & C).sum(axis=0),
    "agreement": (H == C).mean(axis=0).round(2),
}, index=cats)
display(per_cat)

# Reviews where the labels differ, to see what Claude gets wrong
pd.set_option("display.max_colwidth", 500)
diff = hand[(H != C).any(axis=1)]
display(diff[["full_text", "hand_labels", "claude_labels"]])

# Average tokens per review, for the full-batch cost estimate
test_calls = usage_log[-len(hand):]
print(f"Average tokens per review: {np.mean([u[0] for u in test_calls]):.0f} input, "
      f"{np.mean([u[1] for u in test_calls]):.0f} output")

Reviews matched to the original text: 30 of 30
Unrecognized hand labels (fix in the CSV if any): none


/var/folders/74/dlxcg23s4dg9wlkbv81qktk80000gn/T/ipykernel_59095/1867569831.py:10: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  hand["hand_labels"] = hand["my_labels"].fillna("none").apply(


Unreadable Claude replies: 0

Cohen's kappa (all 330 review x category decisions): 0.00
Exact match (identical labels for the whole review): 30%
BELOW 0.6: revise the prompt and test on a new set


,hand_yes,claude_yes,both_yes,agreement
greasy_heavy,0,12,0,0.60
breakouts,0,5,0,0.83
not_moisturizing,0,6,0,0.80
irritation,0,3,0,0.90
fragrance,0,4,0,0.87
pilling,0,0,0,1.00
no_results,0,4,0,0.87
price_value,0,2,0,0.93
packaging,0,1,0,0.97
reformulation,0,0,0,1.00


,full_text,hand_labels,claude_labels
0,"Say yes if you love having a greasy film. I took Sephora’s skin quiz. This was one of the products recommended to me. I looked at the rating...yup, pretty good ones. Yet, this product did not work for my oily skin. I used the whole bottle, just to see if it would grow on me. Nope. Honestly, I don’t understand why the word gel is even used in the name. It’s more cream than gel. When I rub it on my face, it leaves a greasy film that stays on all day. If I powder my face, that greasy film will ...",[],"[greasy_heavy, fragrance, no_results]"
1,Too Heavy. Did not like this.....Too heavy and greasy.,[],[greasy_heavy]
2,"Dries out and tightens skin. I received a trial of this complimentary through Influenster. I have normal skin but it did not react well to this. My skin is sensitive and it caused my face to dry out and tighten. I know it was this product, because as soon as I felt dryness I discontinued use of it and only using my normal cleanser and moisturizer. I would not recommend this product as it left a stinging/tingling feeling when applied and dried my skin out very quickly.",[],"[not_moisturizing, irritation]"
3,Broke me out in a day! It’s very sticky and I broke out all over my face within a day. From the reviews I can see that it happens to a lot of other people. Stay away if you have sensitive skin.,[],"[breakouts, greasy_heavy, irritation]"
4,Not for dry skin!. I purchased this for my dry skin. It literally felt like I had put nothing on my face even when I applied quite a fair amount. No change to my skin at all. Probably isn’t the best for dry skin. My skin isn’t even super dry!,[],"[not_moisturizing, no_results]"
5,"not for me, waxy and sits on skin. I was excited to try this product in a sample pack. I used it in the morning for three days, in hopes that my first day’s observations would subside. They did not. This is a waxy, sit on your skin formula, that never seemed to be absorbed by my skin. At the end of the day, I felt like I had to wash my face due to wanting to wash this product off. I had no other changes to my normal routine during this time.",[],[greasy_heavy]
6,"I really wanted to love this. I really wanted to love this, as I’ve been trying to use clean skincare, and have quite a few DE products. Sadly, every time I add Protini, I end up breaking out. My skin isn’t overly sensitive and can handle actives, but for some reason this just simply doesn’t work for me. I’ve tried it alone, every other day, cocktail with LaLa, and regardless of all of these, I wake up with breakouts. Will probably have to use as a neck cream. Total bummer.",[],[breakouts]
7,Worst break-outs ever. When I first started using this product I loved the formula. My skin was soft and moisturized. Over time though I had really bad sustained acne. I figured it was something else so I didn’t change my moisturizer. I started using a different moisturizer and my skin cleared up. I tried this one again later then the same deep acne came back.,[],[breakouts]
8,This doesn’t sink into my skin quick enough and it doesn’t hydrate enough. I regret buying it,[],"[greasy_heavy, not_moisturizing]"
9,"I received this product from influenster to try. I don’t necessarily dislike this product, but I don’t love it and I don’t really think I’d ever buy it. Using it in the morning in place of my regular moisturizer it felt too sticky after application and then tightened. When I switched it to my night routine it didn’t feel moisturizing enough.",[],"[greasy_heavy, not_moisturizing]"


Average tokens per review: 708 input, 22 output


**Observations**

Claude passes the agreement threshold! On 30 blind hand-labeled reviews (6 per star rating), Cohen's kappa is **0.87** across all 330 review-by-category decisions, well above the 0.6 threshold and in the range conventionally described as almost perfect agreement. Claude's labels exactly matched the hand labels for **80%** of reviews (24 of 30). The full sample is therefore labeled with this prompt unchanged.

**Where Claude and my hand labels differ (6 reviews):** Claude tends to add labels rather than miss them (6 extra labels vs. 2 missed):
- Over-applying "no results": when a reviewer says a product "doesn't work" as a summary of other complaints already labeled (greasiness, breakouts), Claude sometimes adds no_results as well (rows 10 and 13).
- Reading praise or warnings as complaints: "the only good thing… is the super cute packaging" was labeled packaging (row 10), and "stay away if you have sensitive skin" was labeled irritation, although no reaction was described (row 3).
- An extra "other" label: on a review whose complaints were already covered (row 0).
- Missed labels: "didn't WOW me" was hand-labeled no_results but not by Claude (row 12), and "sticky/tight" was hand-labeled not_moisturizing but Claude labeled only greasy_heavy (row 22). Both are borderline judgment calls.

**Claude caught a complaint the hand labeling missed:** in row 22, "wouldn't spend the $50+ that Sephora wants for it" is a price complaint that Claude labeled and the hand labels did not. Hand labels were left unchanged after seeing Claude's results to keep the test blind, so this counts against Claude's score even though Claude was right.

**Implications for the analysis:** Agreement is perfect or near-perfect for the categories most relevant to skin-type fit (greasy_heavy, breakouts, not_moisturizing, fragrance). Because Claude adds no_results, irritation, and other more readily than a human labeler, rates for those categories should be read as upper bounds Rare categories (pilling, reformulation, packaging) barely appear in 30 reviews, so this test says little about their accuracy.

**Fixes made during testing:**
- Truncated reply: in the first run, 1 of 30 replies was unreadable because Claude explained its reasoning before the JSON and hit the 100-token limit mid-answer. The limit was raised to 300 tokens, and the parser now reads the last JSON object in the reply. With both fixes, 0 of 30 replies were unreadable, and kappa was unchanged (0.87).
- Run-to-run variation: current Claude models don't allow setting temperature, so the same review can receive different labels on different runs. Between the two runs, 3 of 30 reviews received different labels, while overall agreement stayed at 0.87. Labels should be read as reliable in aggregate, with some variation at the level of individual reviews.

**Cost per review:** about 708 input tokens (mostly the labeling instructions) and 25 output tokens.

## 4. Full Labeling (Batch API)

All 8,000 sampled reviews are labeled through the Batch API using the same prompt, settings, and parser tested above. The Batch API costs 50% less than real-time calls in exchange for processing within 24 hours. Any reply that fails or can't be read is retried once in real time. Labels are saved to `Data/claude_labels.csv`, so the final report uses saved results instead of calling the API again.

In [40]:
BATCH_ID_PATH = "../Data/batch_id.txt"


def label_params(text):
    """Request settings for labeling one review, matching the settings used in the test above."""
    return {"model": MODEL,
            "max_tokens": LABEL_MAX_TOKENS,
            "thinking": {"type": "disabled"},
            "system": LABEL_SYSTEM_PROMPT,
            "messages": [{"role": "user", "content": text}]}


# Submit only if no batch has been submitted yet, so rerunning this cell never pays twice
if os.path.exists(BATCH_ID_PATH):
    batch_id = open(BATCH_ID_PATH).read().strip()
    print("Batch already submitted:", batch_id)
else:
    # One request per review; custom_id "r00042" links each result back to sample row 42
    requests = [{"custom_id": f"r{i:05d}", "params": label_params(text)}
                for i, text in sample["full_text"].items()]
    batch = client.messages.batches.create(requests=requests)
    batch_id = batch.id
    with open(BATCH_ID_PATH, "w") as f:
        f.write(batch_id)
    print(f"Submitted batch {batch_id} with {len(requests):,} requests")

Submitted batch msgbatch_01VARhF2Zz2CT9bc8ADdAbJ6 with 8,000 requests
